# Music by Genre — Inference

Classify the genre of a music track using [`sugarblock/music_genres_classification-finetuned-gtzan`](https://huggingface.co/sugarblock/music_genres_classification-finetuned-gtzan), a `facebook/wav2vec2-base-960h` fine-tuned on the **GTZAN** dataset (10 genres).

The model expects mono audio resampled to **16 kHz**. GTZAN clips are ~30 seconds, so we split each full track into 30-second windows, classify every window, and average the probabilities for a per-track genre profile.

**GTZAN genres:** blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.

**Inference tracks (compared below) — same files as the emotion / instrument notebooks:**
- `My Drive/google_colab/Nero - Blame You.mp3`
- `My Drive/google_colab/The Cardigans - My Favourite Game (Ozma bootleg).mp3`

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch torchaudio librosa

## 2. Mount Google Drive

After mounting, `My Drive` is available at `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

AUDIO_DIR = '/content/drive/MyDrive/google_colab'
AUDIO_PATHS = [
    f'{AUDIO_DIR}/Nero - Blame You.mp3',
    f'{AUDIO_DIR}/The Cardigans - My Favourite Game (Ozma bootleg).mp3',
]

## 3. Load the model and processor

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_ID = 'sugarblock/music_genres_classification-finetuned-gtzan'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(MODEL_ID).to(device).eval()

# Genre labels come straight from the model config.
GENRE_LABELS = [model.config.id2label[i] for i in range(model.config.num_labels)]

# Sampling rate the feature extractor expects (16 kHz for wav2vec2)
TARGET_SR = extractor.sampling_rate
# GTZAN clips are ~30 seconds; window full tracks to match.
CLIP_SECONDS = 30
CLIP_SAMPLES = TARGET_SR * CLIP_SECONDS

print('Target sampling rate:', TARGET_SR)
print('Clip length:', CLIP_SECONDS, 's', f'({CLIP_SAMPLES} samples)')
print('Genre labels:', GENRE_LABELS)

## 4. Inference helper

Load a track (mono, 16 kHz), split it into non-overlapping 30-second windows, run
the model on every window, and **average the softmax probabilities** across
windows. The result is a per-track genre distribution plus the per-window top
predictions (so we can see how the genre call shifts over the track).

In [ ]:
import os
import numpy as np
import librosa


def classify(path, batch_size=8):
    """Return (mean_probs, window_top_ids, duration_seconds) for one audio file.

    The track is chopped into non-overlapping 30-second windows; the last,
    shorter window is kept only if it holds at least 5 seconds of audio.
    """
    waveform, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    duration = len(waveform) / sr

    # Split into 30-second windows; drop a tiny trailing remnant (< 5 s).
    windows = []
    for start in range(0, len(waveform), CLIP_SAMPLES):
        clip = waveform[start:start + CLIP_SAMPLES]
        if len(clip) < 5 * TARGET_SR:
            continue
        windows.append(clip)
    if not windows:                      # very short track: use the whole thing
        windows = [waveform]

    all_probs = []
    for i in range(0, len(windows), batch_size):
        batch = windows[i:i + batch_size]
        inputs = extractor(batch, sampling_rate=TARGET_SR, return_tensors='pt',
                           padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        all_probs.append(torch.softmax(logits, dim=-1).cpu())

    probs = torch.cat(all_probs, dim=0)        # (n_windows, n_labels)
    mean_probs = probs.mean(dim=0)             # (n_labels,)
    window_top_ids = probs.argmax(dim=-1)      # (n_windows,)
    return mean_probs, window_top_ids, duration


# Run inference on every track up front.
results = {}
window_preds = {}
for path in AUDIO_PATHS:
    name = os.path.splitext(os.path.basename(path))[0]
    mean_probs, top_ids, duration = classify(path)
    results[name] = mean_probs
    window_preds[name] = top_ids
    print(f'{name} — {duration:.1f}s ({len(top_ids)} windows)')

## 5. Top prediction per track

The single most likely genre (averaged over the whole track) plus the share of
30-second windows where it was the top pick.

In [ ]:
for name, probs in results.items():
    top_id = int(probs.argmax())
    top_ids = window_preds[name]
    window_share = (top_ids == top_id).float().mean().item()
    print(f'{name}')
    print(f'  Predicted genre: {GENRE_LABELS[top_id]} '
          f'({probs[top_id].item():.2%} mean prob, '
          f'top in {window_share:.0%} of windows)\n')

## 6. Side-by-side comparison

Full per-genre mean probabilities for both tracks, sorted by the first track's scores.

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    {name: [probs[i].item() for i in range(len(GENRE_LABELS))]
     for name, probs in results.items()},
    index=GENRE_LABELS,
)
comparison.index.name = 'genre'

# Sort by the first track's probability for easy reading.
comparison = comparison.sort_values(by=comparison.columns[0], ascending=False)

# Show as percentages.
(comparison * 100).round(2).style.format('{:.2f}%')

## 7. Genre timeline

How the per-window top genre changes across each track.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(window_preds), 1, figsize=(11, 3 * len(window_preds)),
                         squeeze=False)
for ax, (name, top_ids) in zip(axes[:, 0], window_preds.items()):
    times = np.arange(len(top_ids)) * CLIP_SECONDS
    ax.scatter(times, top_ids.numpy(), s=30)
    ax.set_yticks(range(len(GENRE_LABELS)))
    ax.set_yticklabels(GENRE_LABELS)
    ax.set_xlabel('Time (s)')
    ax.set_title(name)
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()